## LLM With Agentic Loop And Message History

In [10]:
import os
import inspect
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field, SecretStr, BaseModel, ValidationError, create_model
import json 
from functools import wraps
import numexpr as ne
from typing import Callable, Any, Dict
from verbose_utils import (print_separator, print_title,print_info_before_llm_call, print_raw_llm_message, print_memory,
    print_tool_execution, print_memory_state_after_assistant_message, print_memory_state_after_tool_request, print_tool_call_process,
    print_memory_state_after_tool_call
)

In [3]:
class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_file="../.env")
    groq_api_key: SecretStr
    openai_api_key: SecretStr

settings = AppSettings()
print(settings.groq_api_key)   


**********


### The expected structure from LLM
* City name along with weather information.
* Fahrenheit instead of Celsius.

# Langchain Style Tool Creation Decorator

In [4]:
class FuncationMetadataTool:
    """Wraps a Python function with metadata for an LLM."""
    def __init__(self, func: Callable, name: str, description: str, args_schema: type[BaseModel]):
        self.func = func
        self.name = name
        self.description = description
        self.args_schema = args_schema

    def __call__(self, *args, **kwargs) -> Any:
        # Validates arguments against the Pydantic schema before execution
        validated_args = self.args_schema(**kwargs)
        return self.func(**validated_args.model_dump())

    def get_llm_schema(self) -> Dict[str, Any]:
        """Generates OpenAI-style tool definition schema."""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.args_schema.model_json_schema()
            }
        }

def tool(func: Callable) -> FuncationMetadataTool:
    """Decorator to transform a function into a CustomTool."""
    # Extract name and description
    name = func.__name__
    description = func.__doc__ or "No description provided."
    
    # Extract function signatures and type hints
    sig = inspect.signature(func)
    fields = {}
    
    for param_name, param in sig.parameters.items():
        if param_name == 'self':
            continue
        # Default to Any if no type hint is provided
        param_type = param.annotation if param.annotation != inspect.Parameter.empty else Any
        # Handle default values
        default_value = param.default if param.default != inspect.Parameter.empty else ...
        fields[param_name] = (param_type, default_value)
    
    # Dynamically create a Pydantic model for input validation
    schema_name = f"{name}"
    args_schema = create_model(schema_name, **fields)
    
    return FuncationMetadataTool(func, name, description, args_schema)


### Dummy Weather and Calculator Tool.

In [5]:
@tool
def get_weather_information(city: str):
    """
    Retrieve weather information for a supported city.
    Args:
        city (str): The name of the city.
    Returns:
        dict: A dictionary containing:
            - celsius (int): Temperature in degrees Celsius.
            - conditions (str): A brief description of the weather.
    """
    weather = {
        "tokyo": {"celsius": 22, "conditions": "partly cloudy"},
        "delhi": {"celsius": 34, "conditions": "clear skies"},
        "london": {"celsius": 15, "conditions": "light rain"},
    }
    return weather.get(city.lower())

@tool
def calculator(expression: str) -> str:
    """
    Calculates mathematical expressions using numexpr.
    
    Args:
        expression: A string mathematical expression (e.g., "5.6 * (5 + 10.5)").
        
    Returns:
        The result of the calculation as a string.
    """
    try:
        result = ne.evaluate(expression)
        return f"The result of '{expression}' is {result}"
    except Exception as e:
        return f"Error evaluating expression: {e}"

TOOL_SCHEMAS = [get_weather_information.get_llm_schema(), calculator.get_llm_schema()]
print(json.dumps(TOOL_SCHEMAS, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "get_weather_information",
      "description": "\nRetrieve weather information for a supported city.\nArgs:\n    city (str): The name of the city.\nReturns:\n    dict: A dictionary containing:\n        - celsius (int): Temperature in degrees Celsius.\n        - conditions (str): A brief description of the weather.\n",
      "parameters": {
        "properties": {
          "city": {
            "title": "City",
            "type": "string"
          }
        },
        "required": [
          "city"
        ],
        "title": "get_weather_information",
        "type": "object"
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "calculator",
      "description": "\nCalculates mathematical expressions using numexpr.\n\nArgs:\n    expression: A string mathematical expression (e.g., \"5.6 * (5 + 10.5)\").\n\nReturns:\n    The result of the calculation as a string.\n",
      "parameters": {
        

In [6]:
TOOLS_BY_NAME = {"get_weather_information": get_weather_information, 'calculator': calculator}

### LLM with Tools Enabled
* Sends the question plus the tool schema in one call. 
* The reply may contain content or tool_calls list instead suggested by LLM.

### The agentic loop
* In each iteration, call the model with the attached tool.
* If it replies with tool_calls, execute every one of them in same iteration and feed the results back to LLM.
* If it replies with plain content instead, that is the final answer and the loop stops. 
* max_turns is a safety limit so a confused model can't loop forever.

In [7]:
def get_client_and_model():
    from openai import OpenAI
    return OpenAI(api_key=settings.openai_api_key.get_secret_value()), "gpt-4o-mini"


def run_agent(messages: list, max_turns: int = 4, verbose=True) -> str:
    client, model = get_client_and_model()
    
    for turn_number in range(1, max_turns + 1):
        if verbose:
            print_info_before_llm_call(turn_number, messages, TOOL_SCHEMAS, model)

        response = client.chat.completions.create(
            model=model, max_tokens=300, messages=messages, tools=TOOL_SCHEMAS
        )
        message = response.choices[0].message
        
        if verbose:
            print_raw_llm_message(message)

        if not message.tool_calls:
            messages.append({"role": "assistant", "content": message.content})
            if verbose:
                print_memory_state_after_assistant_message(messages)
            return message.content

        messages.append(
            {
                "role": "assistant",
                "content": message.content,
                "tool_calls": [
                    {
                        "id": call.id,
                        "type": "function",
                        "function": {
                            "name": call.function.name,
                            "arguments": call.function.arguments,
                        },
                    }
                    for call in message.tool_calls
                ],
            }
        )

        if verbose:
            print_memory_state_after_tool_request(messages)
        
        for call in message.tool_calls:
            tool_name = call.function.name
            raw_arguments = call.function.arguments
            arguments = json.loads(raw_arguments)
            
            if verbose:
                print_tool_call_process(call, TOOLS_BY_NAME)
            
            tool_function = TOOLS_BY_NAME[tool_name]
            result = tool_function(**arguments)
            
            if verbose:
                print_tool_execution(tool_name, arguments, result)
            
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})
            
            if verbose:
                print_memory_state_after_tool_call(messages)
    return "Reached max_turns without a final answer."


In [8]:
conversation_memory: list[dict] = []

## Demo Conversation With Agent
### 1. Normal LLM call

In [11]:
print_separator()
user_input = 'Hi, I am Shivam'
conversation_memory.append({"role": "user", "content": user_input})
print_title("New user message appended to memory")
print_memory(conversation_memory, "Memory BEFORE running the agent")

answer = run_agent(conversation_memory)
print("\nAgent final answer:")
print(answer)

<---------------------------------->

New user message appended to memory

Memory BEFORE running the agent
Memory contains 1 message(s)

[0] ROLE: user
CONTENT:
Hi, I am Shivam
----------------------------------------------------------------------
<---------------------------------->

AGENT LOOP TURN 1

Memory BEFORE API call
Memory contains 1 message(s)

[0] ROLE: user
CONTENT:
Hi, I am Shivam
----------------------------------------------------------------------

Available tool schemas
1. NAME: get_weather_information
   DESCRIPTION: 
Retrieve weather information for a supported city.
Args:
    city (str): The name of the city.
Returns:
    dict: A dictionary containing:
        - celsius (int): Temperature in degrees Celsius.
        - conditions (str): A brief description of the weather.

   ARGUMENTS:
{
  "properties": {
    "city": {
      "title": "City",
      "type": "string"
    }
  },
  "required": [
    "city"
  ],
  "title": "get_weather_information",
  "type": "object"
}


### Make LLM Tool Call(s).

In [12]:
print_separator()
user_input = 'What is the current weather in capital of japan And solve eqation (8+4)/6+4'
conversation_memory.append({"role": "user", "content": user_input})
print_title("New user message appended to memory")
print_memory(conversation_memory, "Memory BEFORE running the agent")

answer = run_agent(conversation_memory)
print("\nAgent final answer:")
print(answer)

<---------------------------------->

New user message appended to memory

Memory BEFORE running the agent
Memory contains 3 message(s)

[0] ROLE: user
CONTENT:
Hi, I am Shivam
----------------------------------------------------------------------
[1] ROLE: assistant
CONTENT:
Hello Shivam! How can I assist you today?
----------------------------------------------------------------------
[2] ROLE: user
CONTENT:
What is the current weather in capital of japan And solve eqation (8+4)/6+4
----------------------------------------------------------------------
<---------------------------------->

AGENT LOOP TURN 1

Memory BEFORE API call
Memory contains 3 message(s)

[0] ROLE: user
CONTENT:
Hi, I am Shivam
----------------------------------------------------------------------
[1] ROLE: assistant
CONTENT:
Hello Shivam! How can I assist you today?
----------------------------------------------------------------------
[2] ROLE: user
CONTENT:
What is the current weather in capital of japan And 

### Call To Check if LLM Responds Using Conversation History

In [13]:
print_separator()
user_input = 'Who am I?'
conversation_memory.append({"role": "user", "content": user_input})
print_title("New user message appended to memory")
print_memory(conversation_memory, "Memory BEFORE running the agent")

answer = run_agent(conversation_memory)
print("\nAgent final answer:")
print(answer)

<---------------------------------->

New user message appended to memory

Memory BEFORE running the agent
Memory contains 8 message(s)

[0] ROLE: user
CONTENT:
Hi, I am Shivam
----------------------------------------------------------------------
[1] ROLE: assistant
CONTENT:
Hello Shivam! How can I assist you today?
----------------------------------------------------------------------
[2] ROLE: user
CONTENT:
What is the current weather in capital of japan And solve eqation (8+4)/6+4
----------------------------------------------------------------------
[3] ROLE: assistant
CONTENT: None
TOOL CALLS:
  1. id: call_X4NfkH6QYcpvC95lhBV3lLx0
     name: get_weather_information
     arguments: {"city": "Tokyo"}
  2. id: call_nhjLFnRbAo7ubIxI7DmUVfmO
     name: calculator
     arguments: {"expression": "(8+4)/6+4"}
----------------------------------------------------------------------
[4] ROLE: tool
CONTENT:
{'celsius': 22, 'conditions': 'partly cloudy'}
TOOL CALL ID: call_X4NfkH6QYcpvC95lhBV